In [1]:
%pip install pyannote.audio
%pip install numpy==1.26

Note: you may need to restart the kernel to use updated packages.



In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

HUGGING_FACE_TOKEN = os.getenv("HUGGING_FACE_ACCESS_TOKEN")

In [2]:
# instantiate the pipeline
from pyannote.audio import Pipeline

pipeline = Pipeline.from_pretrained(
  "pyannote/speaker-diarization-3.1",
  token=HUGGING_FACE_TOKEN #type:ignore
)

In [3]:
import torch

# cuda가 사용 가능한 경우 cuda를 사용하도록 설정
if torch.cuda.is_available():
    pipeline.to(torch.device("cuda")) #type:ignore
    print('cuda is available')
else:
    print('cuda is not available')

cuda is not available


In [7]:
# run the pipeline on an audio file
# diarization = pipeline("audio.wav")
#diarization = pipeline("../audio/싼기타_비싼기타.mp3") #type:ignore

import torchaudio
wav_path = "../audio/싼기타_비싼기타.wav"
waveform, sample_rate = torchaudio.load(wav_path)

# pyannote 입력 포맷
audio = {"waveform": waveform, "sample_rate": sample_rate}

diarization = pipeline(audio) #type:ignore

# dump the diarization output to disk using RTTM format
with open("싼기타_비싼기타.rttm", "w", encoding='utf-8') as rttm:
    diarization.speaker_diarization.write_rttm(rttm) #type:ignore


In [8]:
# RTTM을 CSV로 변환
import pandas as pd
rttm_path = "./싼기타_비싼기타.rttm"

df_rttm = pd.read_csv(
    rttm_path,      # rttm 파일 경로
    sep=' ',        # 구분자는 띄어쓰기
    header=None,    # 헤더는 없음
    names=['type', 'file', 'chnl', 'start', 'duration', 'C1', 'C2', 'speaker_id', 'C3', 'C4'] 
)

display(df_rttm)

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4
0,SPEAKER,waveform,1,0.993,5.805,NaN,NaN,SPEAKER_00,NaN,NaN
1,SPEAKER,waveform,1,7.405,3.983,NaN,NaN,SPEAKER_00,NaN,NaN
2,SPEAKER,waveform,1,11.759,4.927,NaN,NaN,SPEAKER_00,NaN,NaN
3,SPEAKER,waveform,1,17.210,10.665,NaN,NaN,SPEAKER_00,NaN,NaN
4,SPEAKER,waveform,1,28.668,1.536,NaN,NaN,SPEAKER_00,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
83,SPEAKER,waveform,1,414.481,2.970,NaN,NaN,SPEAKER_01,NaN,NaN
84,SPEAKER,waveform,1,417.755,3.476,NaN,NaN,SPEAKER_00,NaN,NaN
85,SPEAKER,waveform,1,423.644,0.776,NaN,NaN,SPEAKER_01,NaN,NaN
86,SPEAKER,waveform,1,424.741,3.527,NaN,NaN,SPEAKER_01,NaN,NaN


In [9]:
# start + duration을 end로 변환
df_rttm['end'] = df_rttm['start'] + df_rttm['duration']

display(df_rttm)

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end
0,SPEAKER,waveform,1,0.993,5.805,NaN,NaN,SPEAKER_00,NaN,NaN,6.798
1,SPEAKER,waveform,1,7.405,3.983,NaN,NaN,SPEAKER_00,NaN,NaN,11.388
2,SPEAKER,waveform,1,11.759,4.927,NaN,NaN,SPEAKER_00,NaN,NaN,16.686
3,SPEAKER,waveform,1,17.210,10.665,NaN,NaN,SPEAKER_00,NaN,NaN,27.875
4,SPEAKER,waveform,1,28.668,1.536,NaN,NaN,SPEAKER_00,NaN,NaN,30.204
...,...,...,...,...,...,...,...,...,...,...,...
83,SPEAKER,waveform,1,414.481,2.970,NaN,NaN,SPEAKER_01,NaN,NaN,417.451
84,SPEAKER,waveform,1,417.755,3.476,NaN,NaN,SPEAKER_00,NaN,NaN,421.231
85,SPEAKER,waveform,1,423.644,0.776,NaN,NaN,SPEAKER_01,NaN,NaN,424.420
86,SPEAKER,waveform,1,424.741,3.527,NaN,NaN,SPEAKER_01,NaN,NaN,428.268


In [10]:
df_rttm["number"] = None	# number 열 만들고 None으로 초기화
df_rttm.at[0, "number"] = 0

display(df_rttm)

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,waveform,1,0.993,5.805,NaN,NaN,SPEAKER_00,NaN,NaN,6.798,0
1,SPEAKER,waveform,1,7.405,3.983,NaN,NaN,SPEAKER_00,NaN,NaN,11.388,None
2,SPEAKER,waveform,1,11.759,4.927,NaN,NaN,SPEAKER_00,NaN,NaN,16.686,None
3,SPEAKER,waveform,1,17.210,10.665,NaN,NaN,SPEAKER_00,NaN,NaN,27.875,None
4,SPEAKER,waveform,1,28.668,1.536,NaN,NaN,SPEAKER_00,NaN,NaN,30.204,None
...,...,...,...,...,...,...,...,...,...,...,...,...
83,SPEAKER,waveform,1,414.481,2.970,NaN,NaN,SPEAKER_01,NaN,NaN,417.451,None
84,SPEAKER,waveform,1,417.755,3.476,NaN,NaN,SPEAKER_00,NaN,NaN,421.231,None
85,SPEAKER,waveform,1,423.644,0.776,NaN,NaN,SPEAKER_01,NaN,NaN,424.420,None
86,SPEAKER,waveform,1,424.741,3.527,NaN,NaN,SPEAKER_01,NaN,NaN,428.268,None


In [13]:
df_rttm.info() #type:ignore

<class 'pandas.DataFrame'>
RangeIndex: 88 entries, 0 to 87
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   type        88 non-null     str    
 1   file        88 non-null     str    
 2   chnl        88 non-null     int64  
 3   start       88 non-null     float64
 4   duration    88 non-null     float64
 5   C1          0 non-null      float64
 6   C2          0 non-null      float64
 7   speaker_id  88 non-null     str    
 8   C3          0 non-null      float64
 9   C4          0 non-null      float64
 10  end         88 non-null     float64
 11  number      1 non-null      object 
dtypes: float64(7), int64(1), object(1), str(3)
memory usage: 10.5+ KB


In [15]:
for i in range(1, len(df_rttm)):
    if df_rttm.at[i, "speaker_id"] != df_rttm.at[i-1, "speaker_id"]:
        df_rttm.at[i, "number"] = (df_rttm.at[i-1, "number"]) + 1
    else:
        df_rttm.at[i, "number"] = df_rttm.at[i-1, "number"]

display(df_rttm.head(10)) 

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,waveform,1,0.993,5.805,NaN,NaN,SPEAKER_00,NaN,NaN,6.798,0
1,SPEAKER,waveform,1,7.405,3.983,NaN,NaN,SPEAKER_00,NaN,NaN,11.388,0
2,SPEAKER,waveform,1,11.759,4.927,NaN,NaN,SPEAKER_00,NaN,NaN,16.686,0
3,SPEAKER,waveform,1,17.210,10.665,NaN,NaN,SPEAKER_00,NaN,NaN,27.875,0
4,SPEAKER,waveform,1,28.668,1.536,NaN,NaN,SPEAKER_00,NaN,NaN,30.204,0
5,SPEAKER,waveform,1,32.414,0.759,NaN,NaN,SPEAKER_01,NaN,NaN,33.173,1
6,SPEAKER,waveform,1,33.545,3.561,NaN,NaN,SPEAKER_01,NaN,NaN,37.106,1
7,SPEAKER,waveform,1,37.628,3.763,NaN,NaN,SPEAKER_01,NaN,NaN,41.391,1
8,SPEAKER,waveform,1,41.611,0.844,NaN,NaN,SPEAKER_00,NaN,NaN,42.455,2
9,SPEAKER,waveform,1,41.645,1.080,NaN,NaN,SPEAKER_01,NaN,NaN,42.725,3


In [16]:
df_rttm_grouped = df_rttm.groupby("number").agg(
    start=pd.NamedAgg(column='start', aggfunc='min'),
    end=pd.NamedAgg(column='end', aggfunc='max'),
    speaker_id=pd.NamedAgg(column='speaker_id', aggfunc='first')
)

display(df_rttm_grouped)

,start,end,speaker_id
number,,,
0,0.993,30.204,SPEAKER_00
1,32.414,41.391,SPEAKER_01
2,41.611,42.455,SPEAKER_00
3,41.645,42.725,SPEAKER_01
4,42.674,44.024,SPEAKER_00
5,45.813,67.109,SPEAKER_01
6,67.227,82.786,SPEAKER_00
7,84.659,102.564,SPEAKER_01
8,103.492,117.532,SPEAKER_00


In [17]:
df_rttm_grouped["duration"] = df_rttm_grouped["end"] - df_rttm_grouped["start"]
df_rttm_grouped = df_rttm_grouped.reset_index(drop=True)
display(df_rttm_grouped)

,start,end,speaker_id,duration
0,0.993,30.204,SPEAKER_00,29.211
1,32.414,41.391,SPEAKER_01,8.977
2,41.611,42.455,SPEAKER_00,0.844
3,41.645,42.725,SPEAKER_01,1.080
4,42.674,44.024,SPEAKER_00,1.350
5,45.813,67.109,SPEAKER_01,21.296
6,67.227,82.786,SPEAKER_00,15.559
7,84.659,102.564,SPEAKER_01,17.905
8,103.492,117.532,SPEAKER_00,14.040
9,119.759,138.676,SPEAKER_01,18.917


In [18]:
df_rttm_grouped.to_csv(
    "싼기타_비싼기타_rttm.csv",
    sep=',',
    index=False
)